# Лекция 2. Куча: как быстро брать минимум снова и снова

> Конспект второго занятия курса «Алгоритмы и структуры данных» (ДПО).
> Лекция начинается с продолжения прошлой темы: три задачи на бинарный поиск по ответу.
> Из них вырастает один вопрос, который держит всю лекцию: как быстро брать минимум снова и снова.
> Ответ на него это куча: дерево, уложенное в обычный список.
> В конце куча даёт сортировку и возвращает к задаче, с которой вопрос начался.

**После лекции вы сможете:**

- решить задачу бинарным поиском по ответу, когда меняется только проверка кандидата;
- объяснить, что такое куча, и найти в списке предка и потомков любого элемента;
- написать `sift_up`, `sift_down`, `add`, `extract` и объяснить, почему каждая операция стоит `log2(n)` шагов;
- заменить свою кучу модулем `heapq` и отсортировать массив кучей.

**Что нужно знать:** лекция 1: бинарный поиск по ответу, четыре признака, `log2`, инвариант. Python: функции и списки. Функция, которая вызывает сама себя, объясняется в разделе 6.

**Время:** 40 минут на чтение. Около трёх часов, если запускать весь код и делать все предсказания. Лучше за два раза: сначала разделы 1–9, потом разделы 10–16. Если вернулись после перерыва, сначала перезапустите ячейки раздела 3 (импорты и `demo_values`), разделов 6 и 7 (функции кучи) и ячейку с `from heapq import *` из раздела 9.

**Как читать.** Ячейки запускайте по порядку, сверху вниз. Блоки «Предскажите» и «Попробуйте сами» это барьеры: сначала отвечаете сами, потом открываете спойлер. Решения задач E, F, G читают ввод через `input()`, как в Яндекс Контесте: запустите ячейку и наберите строки из условия. Решения задач H и I читают команды из файла `input.txt`. Файл создают ячейки, которые начинаются со строки `%%writefile input.txt`: всё, что ниже этой строки, попадает в файл рядом с ноутбуком. Таких ячеек три: примеры 1 и 2 в разделе 8 и снова пример 1 в разделе 13. Каждая перезаписывает файл целиком.

---

## 1. Продолжение прошлого занятия: три задачи на поиск по ответу

В лекции 1 бинарный поиск искал сторону доски для дипломов. Признаков такой задачи четыре: есть шкала ответов, есть дешёвая проверка одной точки, ответы проверки монотонны, шкала слишком велика для перебора. На втором занятии тем же приёмом решили ещё три задачи. Следите за одним: что меняется в коде от задачи к задаче. К задаче F лекция вернётся дважды. В разделе 2 из неё вырастет главный вопрос, в разделе 14 на неё ответит куча.

### Задача E: два ксерокса

> **Задача E. Ксерокопирование.** Голосовой ассистент Маруся хочет освоить функцию ксерокопирования, причём делать это самым оптимальным образом. У Маруси есть одностраничный документ в одном экземпляре, и ей необходимо сделать ещё `N` копий. В распоряжении Маруси два ксерокса, один из которых копирует лист за `x` секунд, а другой за `y`. Разрешается использовать как один ксерокс, так и оба одновременно. Можно копировать не только с оригинала, но и с копии. Какое минимальное время потребуется Марусе, чтобы сделать ещё `N` копий?
> **Ввод.** Три натуральных числа `N`, `x` и `y`, разделённые пробелом (`1 ≤ N ≤ 2 · 10^8`, `1 ≤ x, y ≤ 10`).
> **Вывод.** Одно число, минимальное время в секундах, необходимое для получения `N` копий.
> **Ограничения.** 1 секунда, 64 МБ.
>
> Примеры: ввод `4 1 1`, вывод `3`. Ввод `5 1 2`, вывод `4`. Ввод `8 3 4`, вывод `15`.

Наивно: прожить процесс по секундам. При `N = 2 · 10^8` и `x = y = 10` ответ равен миллиарду секунд, и цикл по секундам в лимит не уложится.

**Попробуйте сами, прежде чем читать дальше.** Пройдите по четырём признакам. Что здесь шкала, что проверка, откуда монотонность? И почему оба ксерокса нельзя включить с первой секунды?

---

<details><summary>Разбор</summary>

Оригинал один, поэтому первую копию делает один ксерокс. Выгоднее взять быстрый: это `min(x, y)` секунд. В этот момент листов становится два, и оба ксерокса стартуют одновременно.

Дальше шкала это время `m` в секундах после первой копии. Проверка: за время `m` первый ксерокс делает `m // x` копий, второй `m // y`. Они работают одновременно, поэтому копии складываются. Монотонность: чем больше времени, тем больше копий. Ищем наименьшее `m`, за которое два ксерокса делают оставшиеся `N − 1` копий.
</details>

---

Код с занятия. Число копий `N` в нём названо `key`, как искомое число в лекции 1: это порог, с которым сравнивают проверку.

In [ ]:
# введите «8 3 4»; ответ: 15
key, x, y = map(int, input().split())
t = min(x, y)                           # 1. ПЕРВАЯ КОПИЯ: с оригинала, на быстром ксероксе
key -= 1                                #    осталось сделать на одну копию меньше
left, right = 0, max(x, y) * key        # 2. ГРАНИЦЫ: за left не успеть, за right успеть
while right - left > 1:                 # 3. ПОКА между ними есть целая секунда
    m = (left + right) // 2             # 4. СЕРЕДИНА: кандидат во время
    am = m // x + m // y                # 5. ПРОВЕРКА: сколько копий сделают два ксерокса за m
    if am < key:
        left = m
    else:
        right = m
print(t + right)                        # 6. ОТВЕТ: первая копия плюс время на остальные

Правая граница взята по худшему случаю. Даже один медленный ксерокс делает `key` копий за `max(x, y) * key` секунд. Все шаги для ввода `8 3 4`, где после первой копии осталось `key = 7`:

| left | right | m | копий за m | действие |
|---|---|---|---|---|
| 0 | 28 | 14 | `4 + 3 = 7` | `right = 14` |
| 0 | 14 | 7 | `2 + 1 = 3` | `left = 7` |
| 7 | 14 | 10 | `3 + 2 = 5` | `left = 10` |
| 10 | 14 | 12 | `4 + 3 = 7` | `right = 12` |
| 10 | 12 | 11 | `3 + 2 = 5` | `left = 11` |
| 11 | 12 | | | стоп, ответ `3 + 12 = 15` |

**Предскажите.** Что напечатает программа для ввода `1 3 4`? Сколько раз выполнится тело цикла?

---

<details><summary>Ответ</summary>

`3`, и тело цикла не выполнится ни разу. После `key -= 1` остаётся ноль копий, правая граница равна `4 · 0 = 0`, и условие `right - left > 1` ложно сразу. Ответ это одна первая копия на быстром ксероксе. Если вы ждали ошибку на нулевой границе, проверьте инвариант: «за `right = 0` секунд сделать ноль копий можно», он верен.
</details>

---

### Задача F: принтеров сколько угодно

> **Задача F. 3D-принтеры.** Радомир решил открыть небольшой цех по 3D-печати сувениров. У него есть `M` одинаковых по назначению, но разных по скорости 3D-принтеров. Каждый принтер печатает один сувенир за фиксированное время: для `i`-го принтера это `t_i` секунд. В момент времени `0` все принтеры свободны. Как только какой-то принтер заканчивает печать очередного сувенира, его можно сразу же запустить на печать следующего. Все принтеры могут работать одновременно, и каждый принтер может печатать неограниченное количество сувениров один за другим. Радомиру нужно изготовить как минимум `N` одинаковых сувениров. Требуется определить, за какое минимальное время (в секундах) он сможет напечатать не менее чем `N` сувениров, используя свои 3D-принтеры наиболее эффективно.
> **Ввод.** В первой строке числа `N` и `M`, требуемое количество сувениров и количество принтеров (`1 ≤ N ≤ 10^9`, `1 ≤ M ≤ 10`). Во второй строке `M` натуральных чисел `t_i` (`1 ≤ t_i ≤ 10^9`), время печати одного сувенира на каждом из принтеров.
> **Вывод.** Одно целое число, минимальное время в секундах.
> **Ограничения.** 1 секунда, 64 МБ.
>
> Пример 1: строки `1 1` и `5`, вывод `5`. Пример 2: строки `5 1` и `1`, вывод `5`. Пример 3: строки `5 2` и `2 3`, вывод `6`.

Это та же задача E без особой первой копии. Ксероксов было два, принтеров стало `M`. Сумма двух слагаемых превратилась в сумму по списку:

In [ ]:
# введите две строки примера 3: «5 2» и «2 3»; ответ: 6
key, count_printers = map(int, input().split())

times = list(map(int, input().split()))

left, right = 0, max(times) * key               # ГРАНИЦЫ: самый медленный принтер справится один
while right - left > 1:
    m = (left + right) // 2
    cnt = sum(m // t for t in times)            # ПРОВЕРКА: сколько сувениров готово к моменту m
    if cnt < key:
        left = m
    else:
        right = m
print(right)

Запись `sum(m // t for t in times)` читается так: для каждого времени `t` из списка посчитать `m // t` и всё сложить. Число `count_printers` прочитано, но дальше не нужно. Длину знает сам список `times`.

В худшем случае правая граница равна `10^9 · 10^9 = 10^18`. Это те же 60 шагов, что у дипломов в лекции 1: `log2(10^18) ≈ 59.8`.

### Задача G: принтеры печатают партиями

> **Задача G. 3D-принтер (с партиями).** Ариадна открыла свой цех по 3D-печати сувениров с новыми, технологичными принтерами. Каждый такой принтер печатает не по одной штуке за раз, а целую партию. Для каждого принтера известно, что он печатает партию из `c_i` одинаковых сувениров за `t_i` секунд. После того как принтер закончил печатать партию, его можно сразу же запустить на печать следующей такой же партии. Все принтеры могут работать одновременно, и каждый принтер может печатать неограниченное количество партий подряд. Ариадне поступил срочный заказ, нужно изготовить как минимум `N` сувениров. Требуется определить, за какое минимальное время (в секундах) она сможет напечатать не менее чем `N` сувениров, используя свои принтеры наиболее эффективно.
> **Ввод.** В первой строке два целых числа `N` и `M` (`1 ≤ N ≤ 10^9`, `1 ≤ M ≤ 10`). Во второй строке `M` натуральных чисел `c_i` (`1 ≤ c_i ≤ 100`), размер партии на каждом принтере. В третьей строке `M` натуральных чисел `t_i` (`1 ≤ t_i ≤ 10^6`), время печати одной партии.
> **Вывод.** Одно целое число, минимальное время в секундах.
> **Ограничения.** 2 секунды, 512 МБ.
>
> Пример 1: строки `1 1`, `1`, `5`, вывод `5`. Пример 2: строки `10 1`, `2`, `3`, вывод `15`. Пример 3: строки `7 2`, `1 3`, `2 5`, вывод `8`.

Решение G получается из решения F двумя правками. Добавляется чтение размеров партий. А в проверке каждое `m // t` умножается на размер партии `c`: за время `m` принтер успевает `m // t` партий, в каждой `c` сувениров, а незаконченная партия не считается.

In [ ]:
# введите три строки примера 3: «7 2», «1 3», «2 5»; ответ: 8
key, count_printers = map(int, input().split())

counts = list(map(int, input().split()))
times = list(map(int, input().split()))

left, right = 0, max(times) * key
while right - left > 1:
    m = (left + right) // 2
    cnt = sum(m // t * c for c, t in zip(counts, times))    # ПРОВЕРКА: партий m // t, в каждой c штук
    if cnt < key:
        left = m
    else:
        right = m
print(right)

Функция `zip` идёт по двум спискам сразу: первая партия с первым временем, вторая со вторым. Правая граница осталась прежней. В партии не меньше одного сувенира, поэтому самый медленный принтер по-прежнему справится за `max(times) * key`.

Сравните три решения:

| | E, ксероксы | F, принтеры | G, партии |
|---|---|---|---|
| шкала | время `m` | время `m` | время `m` |
| проверка: сколько готово за `m` | `m // x + m // y` | `sum(m // t ...)` | `sum(m // t * c ...)` |
| вне цикла | первая копия отдельно | нет | чтение партий |
| границы, цикл, середина | одинаковые | одинаковые | одинаковые |

Общая форма та же, что в лекции 1: скелет один, меняется только вопрос к середине. Внутри цикла поиска три задачи отличаются одной строкой, проверкой.

В задаче F при этом напрашивается совсем другое решение: не искать ответ, а прожить процесс.

---

## 2. А если моделировать? Откуда берётся вопрос про минимум

Возьмите пример 3 задачи F: нужно пять сувениров, принтеры печатают за 2 и за 3 секунды. Проживите процесс руками. Не по секундам, как в наивном решении задачи E, а по событиям: от сувенира к сувениру. Правило одно: следующий сувенир выдаёт тот принтер, который освободится раньше всех.

| сувенир | принтеры освободятся в | раньше всех | готов в момент |
|---|---|---|---|
| 1 | `2` и `3` | первый | 2 |
| 2 | `4` и `3` | второй | 3 |
| 3 | `4` и `6` | первый | 4 |
| 4 | `6` и `6` | первый | 6 |
| 5 | `8` и `6` | второй | 6 |

Пятый сувенир готов в момент `6`, ответ совпал с бинарным поиском. В каждой строке делается одно и то же. Найти минимум среди моментов, забрать его и положить обратно увеличенным на время печати.

**Предскажите.** Сколько раз придётся «взять минимум и положить обратно» при `N = 10^9`? Уложится ли это в секунду?

---

<details><summary>Ответ</summary>

Миллиард раз, по одному на сувенир. Даже быстрая реализация тратит на это около трёх минут, замер будет в разделе 14. Бинарному поиску хватает 60 шагов, и на каждом он складывает 10 чисел. Поэтому в задаче F побеждает поиск по ответу.
</details>

---

Моделирование проиграло из-за числа сувениров, а не из-за самой операции. Операция «снова и снова взять минимум» никуда не делась. Следующая задача просит только её, двести тысяч раз.

---

## 3. Задача H и два наивных хранилища

> **Задача H. Куча: выбрать минимум.** Необходимо реализовать структуру данных Куча, поддерживающую следующие операции: `CLEAR` сделать кучу пустой; `ADD n` добавить в кучу число `n`; `EXTRACT` удалить из кучи минимальное значение и вывести на экран данное значение. Если куча была пустой, необходимо вывести слово `CANNOT`.
> **Ограничения.** Суммарное количество всех операций не превышает `200000`. 10 секунд, 512 МБ.
>
> Пример 1, ввод:
> ```
> ADD 192168812
> ADD 125
> ADD 321
> EXTRACT
> EXTRACT
> CLEAR
> ADD 7
> ADD 555
> EXTRACT
> EXTRACT
> EXTRACT
> ```
> Вывод: `125`, `321`, `7`, `555`, `CANNOT`.
>
> Пример 2, ввод:
> ```
> CLEAR
> ADD 7
> ADD 555
> EXTRACT
> EXTRACT
> EXTRACT
> CLEAR
> EXTRACT
> ADD 321847
> EXTRACT
> EXTRACT
> ADD 127307
> ADD 949912
> ADD 840884
> ADD 654060
> EXTRACT
> EXTRACT
> ```
> Вывод: `7`, `555`, `CANNOT`, `CANNOT`, `321847`, `CANNOT`, `127307`, `654060`.

Пока забудьте слово «куча» и сделайте в лоб. Числа лежат в списке. Команда `ADD` это `append`, команда `EXTRACT` это `min` и `remove`.

In [ ]:
import random
import time

demo_values = [random.randint(0, 10**9) for _ in range(100000)]

demo_list = list(demo_values)                  # сто тысяч ADD уже сделаны
start = time.perf_counter()
for _ in range(5000):                          # только 5000 EXTRACT из ста тысяч
    smallest = min(demo_list)                  # просмотр всего списка
    demo_list.remove(smallest)                 # и ещё один просмотр, чтобы удалить
print(time.perf_counter() - start)

Печатает около `4.5` секунды, и это пять тысяч извлечений из ста тысяч. Извлечений нужно в двадцать раз больше. Список по дороге укорачивается, в среднем вдвое, поэтому на все сто тысяч уйдёт около 45 секунд при лимите в 10. Каждое извлечение просматривает весь список.

Вторая попытка опирается на лекцию 1. Держите список отсортированным: минимум всегда стоит первым, а место для нового числа находит бинарный поиск. В Python его делает `bisect.insort`:

In [ ]:
import bisect

demo_sorted = []
start = time.perf_counter()
for value in demo_values:
    bisect.insort(demo_sorted, value)          # место ищет бинарный поиск, 17 шагов
for _ in range(100000):
    demo_sorted.pop(0)                         # минимум первый
print(time.perf_counter() - start)

Печатает около `1.2` секунды, и задачу H это проходит. Но посмотрите, как время растёт вместе с количеством чисел:

| чисел в списке (операций вдвое больше) | отсортированный список |
|---|---|
| 100 000 | 1,2 с |
| 200 000 | 5,3 с |
| 400 000 | 21,7 с |

Чисел вдвое больше, времени вчетверо. Бинарный поиск находит место меньше чем за двадцать шагов, но сама вставка сдвигает весь хвост списка на одну позицию. Удаление первого элемента сдвигает вообще всё. Сдвиг в Python быстрый, поэтому на ста тысячах чисел беды нет. На миллионе такой список работает уже минуты.

Отсортированный список держит полный порядок. Про любые два числа известно, какое левее. Задаче H столько не нужно, ей нужен только минимум. Нужна структура с порядком послабее. Чтобы её описать, понадобятся три слова.

---

## 4. Слова для новой структуры: граф, дерево, глубина

Эта структура дерево. Дерево определяют через граф, поэтому начнём с него.

**Граф** (graph) это пара `G = {V, E}`. `V` это множество вершин (vertices), `E` это множество рёбер (edges), а каждое ребро соединяет две вершины. Схема метро это граф: станции вершины, перегоны рёбра.

**Дерево** (tree) это граф, в котором нет замкнутых маршрутов, а из любой вершины можно дойти до любой другой. Кольцевая линия метро это замкнутый маршрут, в дереве такого не бывает. Вот дерево из семи вершин:

```
уровень 1:          A            A это корень
                  /   \
уровень 2:       B     C         B это предок для D и E
                / \   / \
уровень 3:     D   E F   G       D и E это потомки B; D, E, F, G это листья
```

У дерева выделяют одну вершину, **корень** (root), и рисуют его сверху. Соседи вершины уровнем ниже это её **потомки** (children), сосед уровнем выше это **предок** (parent). В коде предок называется `parent`. В других текстах встретите слова «родитель» и «дети», это то же самое. Вершины без потомков называют **листьями** (leaves).

**Бинарное дерево** (binary tree) это дерево, где у каждой вершины не больше двух потомков. **Глубина** `h` (depth), её ещё называют **высотой** дерева (height), это число уровней, на рисунке глубина равна трём.

**Предскажите.** На первом уровне бинарного дерева одна вершина, на втором две, на третьем четыре. Сколько уровней нужно, чтобы поместить 200 000 вершин, если заполнять уровни без пропусков?

---

<details><summary>Ответ</summary>

Восемнадцать. Каждый уровень вдвое шире предыдущего, и `h` заполненных уровней вмещают `2^h − 1` вершин. Семнадцать уровней это `2^17 − 1 = 131 071` вершина, этого мало. Восемнадцать уровней это `2^18 − 1 = 262 143`, этого хватает.

На занятии это записали формулой `h = log(n)`: глубина равна двоичному логарифму числа элементов. Формула даёт порядок величины. Точно так: `log2(200 000) ≈ 17.6`, а уровней целое число, целая часть плюс один, то есть 18.

Это тот же логарифм, что в лекции 1. Там `log2(n)` считал, сколько раз `n` делится пополам. Здесь он считает, сколько раз уровень удваивается, пока дерево не вместит `n`.
</details>

---

Вот почему дерево интересно. В списке из 200 000 чисел путь от начала до конца это 200 000 шагов. В бинарном дереве из 200 000 вершин, где уровни заполнены без пропусков, путь от корня до листа это 17 переходов. Осталось потребовать от дерева ровно столько порядка, сколько нужно для минимума.

---

## 5. Куча: соотношение порядка и дерево в списке

Потребуем одного: каждый предок меньше своих потомков. На занятии это записали так. Для предка `a_i` и его потомков `a_j`, `a_k` выполняются условия `a_i < a_j` и `a_i < a_k`. Это называют **соотношением порядка** (heap property).

Вот дерево из шести чисел, где оно выполнено:

```
          10
        /    \
      30      20
     /  \    /
   50    90 80
```

**Предскажите.** Где в таком дереве минимум? И обещает ли соотношение порядка что-нибудь про пару `30` и `20`? А про пару `50` и `80`?

---

<details><summary>Ответ</summary>

Минимум всегда в корне. Корень меньше своих потомков, те меньше своих, и так до листьев.

Про `30` и `20` соотношение ничего не обещает: они не лежат на одном пути от корня. Здесь `30` стоит левее `20`, и это не нарушение. То же с `50` и `80`: они висят в разных ветках, и могло быть наоборот. Порядок есть только вдоль пути от корня к листу. Это и есть «порядок послабее» из раздела 3.
</details>

---

Второе требование касается формы. Дерево заполняется по уровням, слева направо, без пропусков. Незаполненным может быть только последний уровень, и только справа. Такое дерево называют **почти полным** (almost complete, в англоязычных текстах чаще complete binary tree). Выше именно оно: третий уровень заполнен слева, справа одного листа не хватает.

**Куча** (heap) это почти полное бинарное дерево, в котором выполняется соотношение порядка. Если предок меньше потомков, это куча на минимум (Min-Heap).

Почти полная форма даёт две вещи. Первая: глубина остаётся 18 при 200 000 чисел, дерево не вытягивается в длинную цепочку. Вторая неожиданнее: такое дерево можно хранить в обычном списке. Выпишите вершины по уровням, слева направо:

| индекс | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| значение | 10 | 30 | 20 | 50 | 90 | 80 |

Та же куча картинкой. Под каждой вершиной и под каждой ячейкой стоит её индекс.

![Куча в дереве и в списке](img/heap_tree.png)

Пропусков в дереве нет, поэтому нет их и в списке. Предка и потомков находит арифметика:

| кто | индекс |
|---|---|
| левый потомок вершины `i` | `2 * i + 1` |
| правый потомок вершины `i` | `2 * i + 2` |
| предок вершины `i` | `(i - 1) // 2` |

Откуда формулы. Перед вершиной `i` в списке стоит `i` вершин. У каждой по два потомка, и вместе с корнем они занимают индексы от `0` до `2 * i`. Значит, потомки самой вершины `i` получают следующие два индекса.

**Предскажите.** Какие потомки у вершины с индексом `1`? Кто предок вершины с индексом `5`? Проверьте по дереву.

---

<details><summary>Ответ</summary>

Индексы потомков `2 · 1 + 1 = 3` и `2 · 1 + 2 = 4`, там лежат числа `50` и `90`. Индекс предка `(5 − 1) // 2 = 2`, там лежит число `20`. В дереве `80` действительно висит под `20`.
</details>

---

Посмотрите на список ещё раз: `[10, 30, 20, 50, 90, 80]`. Он не отсортирован, и не должен быть. Гарантировано одно: на нулевом месте минимум.

Взять минимум теперь один шаг: `heap[0]`. Но команды `ADD` и `EXTRACT` из задачи H меняют список, и каждая может сломать порядок, правда, только в одном месте. Чинить его умеют две функции.

---

## 6. Просеивание вверх: sift_up и add

Новое число можно положить только в одно место: в конец списка. Иначе в дереве появится пропуск. Но в конце оно может оказаться меньше своего предка.

Код с занятия:

In [ ]:
heap = []


def sift_up(index: int) -> None:
    if index == 0:                                       # 1. КОРЕНЬ: выше некуда
        return
    parent = (index - 1) // 2                            # 2. ПРЕДОК
    if heap[index] < heap[parent]:                       # 3. ПРОВЕРКА: порядок нарушен?
        heap[index], heap[parent] = heap[parent], heap[index]    # 4. ОБМЕН с предком
        sift_up(parent)                                  # 5. ТО ЖЕ САМОЕ уровнем выше


def add(value: int) -> None:
    heap.append(value)                                   # в конец, чтобы не было пропуска
    sift_up(len(heap) - 1)                               # и починить путь наверх

Три пояснения к записи. Пометки `index: int` и `-> None` это подсказки типов, на работу кода они не влияют. Строка с двумя парами через запятую меняет два элемента местами. Список `heap` объявлен вне функций, и обе функции работают с ним напрямую.

Цикла в `sift_up` нет. Повторяет сама функция: после обмена число стоит уровнем выше, вопрос к нему тот же, и функция вызывает сама себя с новым индексом. Такой приём называют рекурсией (recursion), а сам подъём **просеиванием вверх** (sift up).

**Предскажите.** Как будет выглядеть список после каждого из шести вызовов `add`? Запишите хотя бы последний, потом запустите ячейку. Сами числа на занятии были другие.

In [ ]:
heap.clear()                          # чтобы ячейку можно было запускать повторно
for value in [50, 30, 80, 10, 90, 20]:
    add(value)
    print(value, heap)

---

<details><summary>Анимация: те же шесть вызовов add</summary>

Синим обведено новое число, оранжевым пара, которую сравнивают. Если новое число меньше предка, они меняются местами, и в дереве, и в списке одновременно.

![Шесть вызовов add](img/add.gif)
</details>

---

Трасса вызова `add(10)`, когда в куче лежит `[30, 50, 80]`:

| вызов | список | index | значение | предок | действие |
|---|---|---|---|---|---|
| append | `[30, 50, 80, 10]` | | | | |
| `sift_up(3)` | `[30, 50, 80, 10]` | 3 | 10 | `heap[1] = 50` | `10 < 50`, обмен |
| `sift_up(1)` | `[30, 10, 80, 50]` | 1 | 10 | `heap[0] = 30` | `10 < 30`, обмен |
| `sift_up(0)` | `[10, 30, 80, 50]` | 0 | 10 | | корень, выход по шагу 1 |

Вызовов три, каждый следующий сделан из предыдущего. Когда `sift_up(0)` выходит, `sift_up(1)` уже нечего делать, он тоже заканчивается, а за ним `sift_up(3)`. Соседняя ветка с числом `80` не тронута вовсе.

Остановок у рекурсии две. Первая: дошли до корня. Вторая: проверка на шаге 3 не прошла, порядок уже в норме, и функция просто заканчивается. Вот вторая, на вызове `add(20)` в кучу `[10, 30, 80, 50, 90]`:

| вызов | список | index | значение | предок | действие |
|---|---|---|---|---|---|
| append | `[10, 30, 80, 50, 90, 20]` | | | | |
| `sift_up(5)` | `[10, 30, 80, 50, 90, 20]` | 5 | 20 | `heap[2] = 80` | `20 < 80`, обмен |
| `sift_up(2)` | `[10, 30, 20, 50, 90, 80]` | 2 | 20 | `heap[0] = 10` | `20 < 10` ложно, стоп |

Вторая остановка срабатывает и на равных числах. В формулировке с занятия сравнение строгое, `a_i < a_j`. Одинаковые числа порядок не нарушают: код меняет местами только при строгом `<`, и два равных числа спокойно стоят предком и потомком.

**Предскажите.** В куче 200 000 чисел. Сколько обменов сделает `add` в худшем случае?

---

<details><summary>Ответ</summary>

Семнадцать. Число поднимается по одному уровню за обмен, уровней 18, от нижнего до корня 17 переходов. Отсортированный список при этом сдвинул бы до 200 000 элементов.
</details>

---

Число шагов `add` не больше глубины дерева, а глубина равна `log2(n)`. Это записывают так: `add` работает за `O(log n)`. Запись читается «число шагов растёт не быстрее логарифма размера». Просмотр всего списка в этой записи стоит `O(n)`.

Вставка чинит порядок снизу. Удаление минимума ломает его сверху.

---

## 7. Просеивание вниз: sift_down и extract

Минимум лежит в корне, забрать его легко. Но без корня дерево развалится на два. На занятии поступили так: на место корня переставили последний элемент списка. Форма дерева цела, пропусков нет. Зато в корне теперь почти наверняка слишком большое число, и его нужно опустить. Это **просеивание вниз** (sift down).

In [ ]:
def sift_down(index: int) -> None:
    left = 2 * index + 1                                 # 1. ПОТОМКИ
    right = 2 * index + 2
    if len(heap) < right:                                # 2. ПОТОМКОВ НЕТ: ниже некуда
        return
    if len(heap) == right:                               # 3. ПОТОМОК ОДИН, левый
        right = left                                     #    считаем, что правый это он же
    i_min = left                                         # 4. МЕНЬШИЙ из двух потомков
    if heap[right] < heap[left]:
        i_min = right
    if heap[index] > heap[i_min]:                        # 5. ПРОВЕРКА: порядок нарушен?
        heap[index], heap[i_min] = heap[i_min], heap[index]      # 6. ОБМЕН с меньшим потомком
        sift_down(i_min)                                 # 7. ТО ЖЕ САМОЕ уровнем ниже


def extract():  # get min
    tmp = heap[0]                                        # минимум в корне
    heap[0] = heap[-1]                                   # последний элемент на место корня
    heap.pop()                                           # список стал короче на один
    sift_down(0)                                         # и починить путь вниз
    return tmp

Имена `left` и `right` здесь значат не то, что в разделе 1. Там это были границы поиска, здесь это индексы левого и правого потомка.

Шаги 2 и 3 выясняют, сколько у вершины потомков. Последний индекс в списке равен `len(heap) - 1`. Вот все три случая на списке из четырёх чисел, том же `[90, 30, 80, 50]`, что в трассе ниже:

| вершина | `left` | `right` | сравнение с `len(heap) = 4` | потомков |
|---|---|---|---|---|
| 0 | 1 | 2 | `4 > 2` | два, индексы `1` и `2` |
| 1 | 3 | 4 | `4 == 4` | один, левый с индексом `3`; индекса `4` в списке нет |
| 2 | 5 | 6 | `4 < 6` | ни одного, индекса `5` тоже нет |

Когда потомок один, правым назначают его же, и сравнение в шаге 4 ничего не меняет.

**Предскажите.** Что напечатают два вызова `extract` на куче из раздела 6? Запишите, потом запустите.

In [ ]:
print(extract(), heap)
print(extract(), heap)

---

<details><summary>Анимация: два вызова extract</summary>

Синим отмечен корень, который забирают, и последний элемент, который встаёт на его место. Список при этом становится короче на одну ячейку. Дальше оранжевым отмечена пара, которую сравнивают при просеивании вниз.

![Два вызова extract](img/extract.gif)
</details>

---

Трасса второго `extract`, из кучи `[20, 30, 80, 50, 90]`. Число `20` уходит, `90` встаёт в корень:

| вызов | список | index | значение | потомки | меньший | действие |
|---|---|---|---|---|---|---|
| перестановка | `[90, 30, 80, 50]` | | | | | |
| `sift_down(0)` | `[90, 30, 80, 50]` | 0 | 90 | `30` и `80` | `30` | `90 > 30`, обмен |
| `sift_down(1)` | `[30, 90, 80, 50]` | 1 | 90 | только `50` | `50` | `90 > 50`, обмен |
| `sift_down(3)` | `[30, 50, 80, 90]` | 3 | 90 | нет | | выход по шагу 2 |

В вызове `sift_down(1)` сработал случай с одним потомком. Остановок снова две: потомков нет, или проверка на шаге 5 не прошла.

**Предскажите.** Зачем в шаге 4 выбирать меньшего потомка? Что сломается, если всегда меняться с левым? Проверьте на куче `[80, 30, 20]`.

---

<details><summary>Ответ</summary>

Обмен с левым даст `[30, 80, 20]`. Теперь в корне `30`, а его правый потомок `20` меньше. Соотношение порядка нарушено, и починить его уже некому. Наверх должен подняться меньший из потомков, потому что он станет предком второго.
</details>

---

Цена та же, что у `add`. Число спускается по одному уровню за обмен, значит, `extract` работает за `O(log n)`.

### Где подведёт: extract из пустой кучи

В куче из одного числа `extract` работает. Корень копируется сам в себя, `pop` оставляет пустой список, и `sift_down(0)` сразу выходит по шагу 2. А из пустой?

**Предскажите.** Что будет, если вызвать `extract()`, когда `heap` пуст?

In [ ]:
heap.clear()
try:
    extract()
except IndexError as error:
    print("IndexError:", error)

---

<details><summary>Ответ</summary>

Функция падает на первой же своей строке: `IndexError: list index out of range`. Корня нет, и `heap[0]` взять неоткуда. Сама функция пустоту не проверяет, поэтому проверять обязан тот, кто её вызывает. В задаче H для этого случая и придумано слово `CANNOT`.
</details>

---

Обе операции готовы, и задача H решается целиком.

---

## 8. Задача H решена

Решение с занятия читает команды из файла `input.txt`: контест разрешает и такой ввод. Эта ячейка создаёт файл с примером 1:

In [ ]:
%%writefile input.txt
ADD 192168812
ADD 125
ADD 321
EXTRACT
EXTRACT
CLEAR
ADD 7
ADD 555
EXTRACT
EXTRACT
EXTRACT

Функции `sift_up`, `sift_down`, `add` и `extract` уже определены в ячейках выше. Остался цикл по командам:

In [ ]:
f = open('input.txt')

heap = []

for line in f:
    line = line.split()
    if line[0] == 'ADD':
        add(int(line[1]))
    elif line[0] == 'EXTRACT':
        if heap:                                # пустую кучу проверяет вызывающий
            print(extract())
        else:
            print("CANNOT")
    elif line[0] == "CLEAR":
        heap.clear()

Печатает `125`, `321`, `7`, `555`, `CANNOT`, как в условии.

Теперь пример 2. Запустите ячейку ниже, она перезапишет файл. Потом ещё раз запустите ячейку с циклом и сверьте вывод с условием.

In [ ]:
%%writefile input.txt
CLEAR
ADD 7
ADD 555
EXTRACT
EXTRACT
EXTRACT
CLEAR
EXTRACT
ADD 321847
EXTRACT
EXTRACT
ADD 127307
ADD 949912
ADD 840884
ADD 654060
EXTRACT
EXTRACT

Ответы верные. Осталась скорость, ради которой всё затевалось. Замер на тех же ста тысячах чисел, на которые списку из раздела 3 нужно около 45 секунд:

In [ ]:
heap = []
start = time.perf_counter()
for value in demo_values:
    add(value)
for _ in range(100000):
    extract()
print(time.perf_counter() - start)

Печатает около `0.35` секунды. Двести тысяч операций, каждая не дороже 16 обменов: в дереве из 100 000 вершин 17 уровней.

| хранилище | 100 000 чисел, 200 000 операций | если чисел вдвое больше |
|---|---|---|
| список, `min` и `remove` | около 45 с | вчетверо дольше |
| отсортированный список | 1,2 с | вчетверо дольше |
| куча | 0,35 с | чуть больше чем вдвое |

Это и есть ответ на вопрос лекции. Сколько бы раз подряд ни просили минимум, каждый раз он стоит не больше глубины дерева.

Такую структуру не пишут каждый раз заново.

---

## 9. Та же куча в стандартной библиотеке: heapq

В Python куча на минимум есть готовая, в модуле `heapq`. Второе решение задачи H с занятия:

In [ ]:
from heapq import *

f = open('input.txt')

heap = []


for line in f:
    line = line.split()
    if line[0] == 'ADD':
        heappush(heap, int(line[1]))
    elif line[0] == 'EXTRACT':
        if heap:
            print(heappop(heap))
        else:
            print("CANNOT")
    elif line[0] == "CLEAR":
        heap.clear()

В файле сейчас пример 2, и печатается его вывод из условия: `7`, `555`, `CANNOT`, `CANNOT`, `321847`, `CANNOT`, `127307`, `654060`. Звёздочка в первой строке забирает из модуля все функции сразу: `heappush`, `heappop`, а ещё `heapify`, которая понадобится в разделе 11.

Модуль не заводит нового типа. Куча в нём это обычный список, с той же раскладкой по индексам, что у вас.

| своя куча | `heapq` |
|---|---|
| `add(value)` | `heappush(heap, value)` |
| `extract()` | `heappop(heap)` |
| `heap[0]`, минимум без удаления | `heap[0]` |
| список `heap` объявлен вне функций | список передаётся аргументом |

**Предскажите.** Во сколько раз `heapq` быстрее вашей кучи на тех же 200 000 операций: в полтора, в десять, в тысячу?

In [ ]:
heap = []
start = time.perf_counter()
for value in demo_values:
    heappush(heap, value)
for _ in range(100000):
    heappop(heap)
print(time.perf_counter() - start)

---

<details><summary>Ответ</summary>

Почти в десять раз: около `0.04` секунды против `0.35`. Алгоритм тот же самый, те же просеивания. Разница в том, что `heapq` написан на C. Растут обе одинаково медленно: вдвое больше чисел, времени чуть больше чем вдвое.
</details>

---

В `heapq` есть вставка и извлечение. У кучи операций больше, и все они собираются из тех же двух просеиваний.

---

## 10. Остальные операции: те же два просеивания

Пока числа только добавляли и забирали. Но число, уже лежащее в куче, может измениться или стать ненужным, а брать минимум после этого нужно так же быстро. На занятии для этого назвали ещё три операции: уменьшить значение, увеличить значение, удалить элемент. Для каждой достаточно решить, в какую сторону сломался порядок. Индекс `i` изменяемого элемента нужно знать заранее. Искать число в куче пришлось бы просмотром всего списка, а это уже `O(n)`.

Соберите знакомую кучу и уменьшите число `90` до пяти:

In [ ]:
heap = []
for value in [50, 30, 80, 10, 90, 20]:
    add(value)
print(heap)                 # [10, 30, 20, 50, 90, 80]

heap[4] = 5                 # уменьшили: число могло стать меньше предка
sift_up(4)                  # значит, просеять вверх
print(heap)

Вторая строка вывода: `[5, 10, 20, 50, 30, 80]`. Пятёрка поднялась в корень за два обмена.

| операция | что могло сломаться | чем чинить | цена |
|---|---|---|---|
| уменьшить `heap[i]` | число стало меньше предка | `sift_up(i)` | `O(log n)` |
| увеличить `heap[i]` | число стало больше потомка | `sift_down(i)` | `O(log n)` |
| удалить `heap[i]` | на место `i` встаёт последний элемент списка, он может быть и меньше предка, и больше потомка | `sift_up(i)`, затем `sift_down(i)`; сработает только одно. Если `i` и есть последний индекс, элемент просто снимают `pop` | `O(log n)` |
| добавить | новое число в конце меньше предка | `sift_up` | `O(log n)` |
| взять минимум | последний элемент в корне больше потомка | `sift_down(0)` | `O(log n)` |

**Предскажите.** Ячейка ниже собирает ту же кучу заново, увеличивает корень до `70` и просеивает его вниз. Что напечатается?

In [ ]:
heap = []
for value in [50, 30, 80, 10, 90, 20]:
    add(value)

heap[0] = 70                # увеличили: число могло стать больше потомка
sift_down(0)                # значит, просеять вниз
print(heap)

---

<details><summary>Ответ</summary>

`[20, 30, 70, 50, 90, 80]`. Число `70` меняется с меньшим потомком, `20`, и останавливается: его новый потомок `80` больше.
</details>

---

### Где подведёт: изменить и не починить

**Предскажите.** В куче `[10, 30, 20, 50, 90, 80]` выполнили `heap[4] = 5` и забыли про `sift_up`. Что вернёт `extract()`?

In [ ]:
heap = [10, 30, 20, 50, 90, 80]
heap[4] = 5
print(extract(), heap)

---

<details><summary>Ответ</summary>

`10`, хотя минимум теперь `5`. Пятёрка так и осталась лежать в конце: `[20, 30, 80, 50, 5]`. Функция `extract` не ищет минимум, она верит, что он в корне. Ошибки нет, ответ есть, и он неверный. Это та же беда, что бинарный поиск по неотсортированному массиву в лекции 1: алгоритм опирается на порядок, который никто не проверяет.

Правило: список кучи нельзя менять напрямую. Любое изменение значения должно заканчиваться просеиванием.
</details>

---

Все эти операции меняют готовую кучу. А как получить кучу из готового массива?

---

## 11. Окучивание: куча из массива за O(n)

Очевидный способ: завести пустую кучу и сделать `n` вставок. Каждая стоит до `log2(n)` шагов, всего `O(n · log n)`. На занятии назвали способ быстрее, **окучивание** (heapify), и его цену: `O(n)`. В учебниках эту операцию называют построением кучи (build heap). В `heapq` он есть готовый:

In [ ]:
demo_array = [50, 30, 80, 10, 90, 20]
heapify(demo_array)                # переставляет элементы на месте
print(demo_array)

Печатает `[10, 30, 20, 50, 90, 80]`. Тот же список стал кучей, нового списка не создано.

Идея окучивания: идти по вершинам от последней к первой и каждую просеивать вниз. Вершина вместе со всем, что под ней висит, называется поддеревом. К моменту, когда очередь доходит до вершины, оба поддерева под ней уже кучи, и одного `sift_down` достаточно.

**Предскажите.** Для `[50, 30, 80, 10, 90, 20]`: какие вершины придётся просеивать и в каком порядке?

---

<details><summary>Ответ и анимация</summary>

Индексы `2`, `1`, `0`. Листьям с индексами `3`, `4`, `5` просеиваться некуда, у них нет потомков. Синим кольцом обведена вершина, с которой начинается очередное просеивание, оранжевым пара, которую сравнивают.

![Окучивание по шагам](img/heapify.gif)
</details>

---

Все шаги по порядку:

| просеиваем вниз | список после | что произошло |
|---|---|---|
| | `[50, 30, 80, 10, 90, 20]` | исходный массив |
| индекс `2`, число `80` | `[50, 30, 20, 10, 90, 80]` | обмен с потомком `20` |
| индекс `1`, число `30` | `[50, 10, 20, 30, 90, 80]` | обмен с меньшим потомком `10` |
| индекс `0`, число `50` | `[10, 30, 20, 50, 90, 80]` | обмен с `10`, потом с `30` |

Почему это дешевле `n` вставок. Посчитайте, сколько шагов вниз может сделать вершина:

| вершины | доля от всех | шагов вниз, не больше |
|---|---|---|
| листья | половина | 0 |
| уровень над листьями | четверть | 1 |
| ещё уровнем выше | восьмая | 2 |
| корень | одна вершина | `log2(n)` |

Пример на дереве из 15 вершин, в нём четыре уровня: `8 · 0 + 4 · 1 + 2 · 2 + 1 · 3 = 11` шагов, меньше 15. Половина вершин не двигается вообще, и только одна проходит весь путь. При вставках наоборот: половина чисел попадает на нижний уровень, откуда путь наверх самый длинный. Для миллиона элементов окучивание делает не больше миллиона шагов, а `n · log2(n)` это двадцать миллионов.

**Предскажите.** Миллион чисел идёт по убыванию: `1000000, 999999, …, 1`. Что быстрее превратит их в кучу, миллион `heappush` или один `heapify`, и во сколько раз?

In [ ]:
demo_descending = list(range(10**6, 0, -1))

demo_heap = []
start = time.perf_counter()
for value in demo_descending:
    heappush(demo_heap, value)
print("миллион heappush:", time.perf_counter() - start)

start = time.perf_counter()
heapify(demo_descending)
print("один heapify:   ", time.perf_counter() - start)

---

<details><summary>Ответ</summary>

`heapify` быстрее раз в пятнадцать: около `0.013` секунды против `0.18`. Убывающий вход худший для вставок. Каждое новое число меньше всех прежних и поднимается до самого корня.
</details>

---

Окучивание превращает массив в кучу. А если после этого достать из кучи всё по очереди, элементы выйдут по возрастанию.

---

## 12. Пирамидальная сортировка: n раз взять минимум

Сортировка получается как побочный продукт. Этапов два: окучить массив, потом `n` раз взять минимум.

In [ ]:
demo_array = [50, 30, 80, 10, 90, 20]

heapify(demo_array)                                            # этап 1: окучивание, O(n)
demo_result = [heappop(demo_array) for _ in range(6)]          # этап 2: n извлечений по O(log n)
print(demo_result)

Запись в квадратных скобках собирает список: шесть раз вызвать `heappop` и сложить результаты по порядку. Печатает `[10, 20, 30, 50, 80, 90]`.

Это **пирамидальная сортировка** (heap sort). Пирамида это старое русское название кучи. Цена складывается из двух этапов: `O(n + n · log n)`. Второе слагаемое растёт быстрее первого и поглощает его, поэтому пишут коротко: `O(n · log n)`.

**Предскажите.** Миллион случайных чисел сортируют двумя способами: пирамидальной сортировкой через `heapq` и встроенной `sorted`. Совпадут ли результаты? Кто быстрее?

In [ ]:
demo_million = [random.randint(0, 10**9) for _ in range(10**6)]

demo_copy = list(demo_million)
start = time.perf_counter()
heapify(demo_copy)
demo_by_heap = [heappop(demo_copy) for _ in range(10**6)]
print("кучей: ", time.perf_counter() - start)

start = time.perf_counter()
demo_by_sorted = sorted(demo_million)
print("sorted:", time.perf_counter() - start)

print(demo_by_heap == demo_by_sorted)

---

<details><summary>Ответ</summary>

Результаты совпадут, последняя строка печатает `True`. Встроенная `sorted` быстрее раза в четыре: около `0.2` секунды против `0.85`. Обе сортировки стоят `O(n · log n)`, но запись `O(...)` прячет множитель, а он у разных алгоритмов разный. Она говорит, как время растёт, а не чему оно равно.
</details>

---

Сортировка вышла по возрастанию, потому что куча отдаёт минимум. А если снова и снова нужен максимум? Это следующая задача контеста, и для неё хватит трёх знаков.

---

## 13. А если нужен максимум: задача I

> **Задача I. Куча: выбрать максимум.** Необходимо реализовать структуру данных Куча, поддерживающую следующие операции: `CLEAR` сделать кучу пустой; `ADD n` добавить в кучу число `n`; `EXTRACT` удалить из кучи максимальное значение и вывести на экран данное значение. Если куча была пустой, необходимо вывести слово `CANNOT`.
> **Ограничения.** Суммарное количество всех операций не превышает `200000`. 10 секунд, 512 МБ.
>
> Примеры ввода те же, что в задаче H. Вывод для примера 1: `192168812`, `321`, `555`, `7`, `CANNOT`. Для примера 2: `555`, `7`, `CANNOT`, `CANNOT`, `321847`, `CANNOT`, `949912`, `840884`.

Соотношение порядка переворачивается: предок больше потомков. Такую кучу называют кучей на максимум (Max-Heap). Форма дерева, индексы, `add` и `extract` остаются прежними. Меняются три знака: сравнение с предком в `sift_up`, выбор потомка и сравнение с потомком в `sift_down`. Сравнения с `len(heap)` не трогаются, они про форму, а не про порядок.

Решение снова читает `input.txt`, а там сейчас пример 2. Верните в файл пример 1:

In [ ]:
%%writefile input.txt
ADD 192168812
ADD 125
ADD 321
EXTRACT
EXTRACT
CLEAR
ADD 7
ADD 555
EXTRACT
EXTRACT
EXTRACT

In [ ]:
f = open('input.txt')

heap = []


def sift_up(index: int) -> None:
    if index == 0:
        return
    parent = (index - 1) // 2
    if heap[index] > heap[parent]:                       # было <: наверх идёт большее
        heap[index], heap[parent] = heap[parent], heap[index]
        sift_up(parent)


def sift_down(index: int) -> None:
    left = 2 * index + 1
    right = 2 * index + 2
    if len(heap) < right:
        return
    if len(heap) == right:
        right = left
    i_max = left
    if heap[right] > heap[left]:                         # было <: выбираем большего потомка
        i_max = right
    if heap[index] < heap[i_max]:                        # было >: предок должен быть больше
        heap[index], heap[i_max] = heap[i_max], heap[index]
        sift_down(i_max)


for line in f:
    line = line.split()
    if line[0] == 'ADD':
        add(int(line[1]))
    elif line[0] == 'EXTRACT':
        if heap:
            print(extract())
        else:
            print("CANNOT")
    elif line[0] == "CLEAR":
        heap.clear()

Печатает `192168812`, `321`, `555`, `7`, `CANNOT`. Эта ячейка заменила `sift_up` и `sift_down` версиями на максимум. Чтобы вернуться к минимуму, перезапустите ячейки разделов 6 и 7.

В `heapq` до Python 3.14 кучи на максимум нет, а в Контесте стоит Python 3.13. Обходятся сменой знака: кладут `-value`, достают и снова меняют знак. Числа `7` и `555` лягут в кучу как `-7` и `-555`. Минимум из них это `-555`, и после смены знака выходит `555`, максимум.

In [ ]:
f = open('input.txt')

heap = []

for line in f:
    line = line.split()
    if line[0] == 'ADD':
        heappush(heap, -int(line[1]))           # кладём с минусом
    elif line[0] == 'EXTRACT':
        if heap:
            print(-heappop(heap))               # достаём и возвращаем знак
        else:
            print("CANNOT")
    elif line[0] == "CLEAR":
        heap.clear()

Вывод тот же. Куча теперь отдаёт и минимум, и максимум, а `heapify` строит её из готового списка. Этого хватает, чтобы вернуться к принтерам, с которых вопрос начался.

---

## 14. Возврат к принтерам: моделирование с кучей

В разделе 2 вы прожили пример задачи F руками: пять раз нашли принтер, который освободится раньше всех. Теперь это делает куча. В ней лежат пары «момент готовности, время печати». Python сравнивает пары по первому числу, а при равных первых по второму: `(4, 3) < (6, 2)` и `(6, 2) < (6, 3)`. Поэтому в корне всегда самый ранний момент.

In [ ]:
demo_need, demo_times = 5, [2, 3]

demo_printers = [(t, t) for t in demo_times]             # каждый принтер закончит первый сувенир в момент t
heapify(demo_printers)
for number in range(1, demo_need + 1):
    done, t = heappop(demo_printers)                     # достали пару и разложили её на два имени
    heappush(demo_printers, (done + t, t))               # тот же принтер сразу берёт следующий сувенир
    print("сувенир", number, "готов в момент", done, "на принтере со временем", t)
print(done)

Пять строк вывода повторяют таблицу из раздела 2: моменты `2`, `3`, `4`, `6`, `6`. Последняя строка печатает `6`, как бинарный поиск в разделе 1.

Обещанный замер. Миллион сувениров на десяти принтерах:

In [ ]:
demo_printers = [(t, t) for t in [3, 5, 7, 11, 13, 17, 19, 23, 29, 31]]
heapify(demo_printers)
start = time.perf_counter()
for _ in range(10**6):
    done, t = heappop(demo_printers)
    heappush(demo_printers, (done + t, t))
print(time.perf_counter() - start)

Печатает около `0.2` секунды. Миллиард сувениров в тысячу раз дольше: около трёх минут при лимите в одну секунду.

Сравните два решения одной задачи при `N = 10^9` и десяти принтерах:

| | моделирование с кучей | поиск по ответу |
|---|---|---|
| шагов | `10^9` извлечений, каждое `O(log M)` | 60 середин, в каждой сумма по 10 принтерам |
| время | минуты | доли секунды |
| что умеет сверх ответа | выдаёт расписание: когда готов каждый сувенир | только итоговое время |

Прогноз из раздела 2 подтвердился замером. Вопрос в задаче F только про итоговое время, а сувениров миллиард. Если бы спросили расписание, понадобилось бы моделирование, а при большом числе принтеров и куча.

Куча сделала дешёвым каждое взятие минимума, но не уменьшила их число. Там, где брать приходится миллиард раз, выигрывает тот, кто не берёт вовсе.

---

## 15. Проверьте себя

Ответьте без кода, потом сверьтесь.

1. В задаче G принтер печатает партию из 3 сувениров за 5 секунд. Сколько сувениров он выдаст за 14 секунд и почему не 8?
2. Куча лежит в списке `[20, 40, 30, 90, 50, 70]`. Кто предок числа `50`? Кто потомки числа `30`?
3. Объясните своими словами, почему `add` в кучу из миллиона чисел делает не больше 19 обменов.
4. Список `[10, 20, 30, 40, 50]` отсортирован. Является ли он кучей на минимум? А верно ли обратное: любая куча отсортирована?
5. Почему окучивание стоит `O(n)`, хотя каждое просеивание вниз может стоить `log2(n)` шагов?

---

<details><summary>Ответы</summary>

1. Шесть: `14 // 5 = 2` полные партии по 3 штуки. Третья партия к 14-й секунде не закончена, а незаконченная партия не даёт ни одного сувенира.
2. Число `50` стоит на индексе `4`, индекс его предка `(4 − 1) // 2 = 1`, там лежит `40`. Число `30` стоит на индексе `2`, его потомки на индексах `5` и `6`. На индексе `5` лежит `70`, а индекса `6` в списке нет: потомок один.
3. Новое число встаёт в конец и поднимается по одному уровню за обмен. Уровней в дереве из миллиона вершин двадцать, потому что `2^20` чуть больше миллиона. От нижнего уровня до корня 19 переходов.
4. Да, отсортированный по возрастанию список это куча: каждый предок стоит левее своих потомков, значит, он меньше. Обратное неверно: `[10, 30, 20, 50, 90, 80]` это куча, и она не отсортирована.
5. Потому что длинный путь вниз есть только у вершин наверху, а их мало. Половина вершин это листья с нулём шагов, четверть делает не больше одного шага. Для 15 вершин выходит 11 шагов, и сумма по всем вершинам никогда не превышает `n`.
</details>

---

## 16. Итог

- Поиск по ответу решает целое семейство задач. В E, F и G внутри цикла поиска меняется одна строка: проверка «сколько готово за время `m`». В задаче F он выигрывает у моделирования с кучей, зато моделирование даёт расписание.
- Куча это почти полное бинарное дерево с соотношением порядка, уложенное в список. Предок `(i − 1) // 2`, потомки `2i + 1` и `2i + 2`, минимум на нулевом месте.
- Порядок чинят два просеивания: `sift_up` после вставки и уменьшения, `sift_down` после извлечения и увеличения. Цена любой операции не больше глубины дерева, `O(log n)`. Поэтому минимум можно брать снова и снова: список платит за каждое взятие `O(n)`, куча `O(log n)`.
- Окучивание строит кучу из массива за `O(n)`, а окучивание плюс `n` извлечений это пирамидальная сортировка за `O(n · log n)`. В Python всё это есть в `heapq`.

**Где встретится дальше.** Классификацию алгоритмов и бинарный поиск вы разобрали в лекции 1, кучу в этой. Дальше вас ждут деревья и сортировка с помощью дерева. Слова «вершина», «корень», «потомок», «глубина» (она же высота) и оценка `O(n · log n)` понадобятся там сразу, так что держите их под рукой.

**Что повторить через два-три дня.** Напишите `sift_up` и `sift_down` по памяти и добавьте в кучу числа `50, 30, 80, 10, 90, 20`. Нарисуйте получившийся список деревом. Объясните вслух, почему куча не отсортирована и почему это не мешает брать минимум.